<a href="https://colab.research.google.com/github/SMK-BTW/Twenty/blob/main/Machine_Learning_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q --upgrade --force-reinstall pandas pyarrow datasets huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.6/111.6 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 75.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.0/120.0 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.4/110.4 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.9/203.9 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
!pip install -q duckdb

In [4]:
from huggingface_hub import hf_hub_url
import duckdb

In [5]:
url = hf_hub_url(
    repo_id="yandex/yambda",
    filename="flat/50m/multi_event.parquet",
    repo_type="dataset"
)


In [6]:
con = duckdb.connect()

con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [7]:
df = con.execute(f"""
    SELECT *
    FROM read_parquet('{url}')
    USING SAMPLE 100000 ROWS
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [8]:
print(df.shape)
print("Unique users:", df["uid"].nunique())
print("Unique tracks:", df["item_id"].nunique())

print("\nEvent types:")
print(df["event_type"].value_counts())

(100000, 7)
Unique users: 8094
Unique tracks: 47025

Event types:
event_type
listen       97400
like          1740
unlike         626
dislike        182
undislike       52
Name: count, dtype: int64


In [9]:
df.head()

,uid,timestamp,item_id,is_organic,played_ratio_pct,track_length_seconds,event_type
0,755400,25950795,3617925,0,1,290,listen
1,80000,1459435,7307960,0,99,160,listen
2,344300,22461950,1677342,0,100,185,listen
3,231100,24341980,6629575,1,1,225,listen
4,139100,20362125,6615285,0,2,165,listen


In [10]:
df.isnull().sum()

uid                        0
timestamp                  0
item_id                    0
is_organic                 0
played_ratio_pct        2600
track_length_seconds    2600
event_type                 0
dtype: int64

played_ratio_pct is the percentage of track played (1-100)

In [11]:
listening_df = df[
    (df["event_type"] == "listen") &
    (df["played_ratio_pct"] > 50)
].copy()

In [12]:
print("Meaningful listening events:", len(listening_df))
print("Unique users:", listening_df["uid"].nunique())
print("Unique songs:", listening_df["item_id"].nunique())

Meaningful listening events: 62664
Unique users: 7156
Unique songs: 33651


We have meaningful listen data but the user might have listened to one track several times so we collapse them

In [13]:
interactions = (
    listening_df[
        ["uid", "item_id"]
    ]
    .drop_duplicates()
    .copy()
)

interactions["interaction"] = 1

interactions.head()

,uid,item_id,interaction
1,80000,7307960,1
2,344300,1677342,1
5,906200,6587535,1
6,432400,8055973,1
7,942200,3927123,1


In [14]:
print("User-song pairs:", len(interactions))
print("Users:", interactions["uid"].nunique())
print("Songs:", interactions["item_id"].nunique())

User-song pairs: 60205
Users: 7156
Songs: 33651


In [15]:
user_activity = (
    interactions.groupby("uid")
    .size()
    .sort_values(ascending=False)
)

user_activity.head(10)

uid
7200    1348
1400     850
800      658
1600     532
7700     508
7400     485
700      434
7100     395
1100     351
600      324
dtype: int64

In [16]:
target_user = user_activity.index[0]

print("Sample target user:", target_user)
print(
    "Number of unique songs:",
    user_activity.loc[target_user]
)

Sample target user: 7200
Number of unique songs: 1348


Sample input Data for the proposal

In [17]:
sample_input = (
    listening_df[
        listening_df["uid"] == target_user
    ][
        [
            "uid",
            "timestamp",
            "item_id",
            "played_ratio_pct",
            "event_type"
        ]
    ]
    .head(10)
)

sample_input

,uid,timestamp,item_id,played_ratio_pct,event_type
229,7200,3108820,3885453,100,listen
246,7200,2548485,1875837,100,listen
255,7200,18877285,3677156,100,listen
288,7200,14188845,7381692,100,listen
439,7200,3398995,2411822,100,listen
450,7200,2403995,5471204,100,listen
554,7200,12931870,5097421,100,listen
751,7200,12485005,5097421,100,listen
837,7200,13769520,1238245,100,listen
886,7200,10043470,7355371,100,listen


K-NN Model Input

In [18]:
sample_model_input = interactions[
    interactions["uid"] == target_user
].copy()

sample_model_input.head(10)

,uid,item_id,interaction
229,7200,3885453,1
246,7200,1875837,1
255,7200,3677156,1
288,7200,7381692,1
439,7200,2411822,1
450,7200,5471204,1
554,7200,5097421,1
837,7200,1238245,1
886,7200,7355371,1
1042,7200,4834547,1


In [19]:
OUTPUT_COLUMNS = [
    "uid",
    "rank",
    "item_id",
    "score"
]

print("Expected recommender output columns:")
print(OUTPUT_COLUMNS)

Expected recommender output columns:
['uid', 'rank', 'item_id', 'score']


In [20]:
already_listened = set(
    interactions.loc[
        interactions["uid"] == target_user,
        "item_id"
    ]
)

print("Target user:", target_user)
print("Songs already listened to:", len(already_listened))

Target user: 7200
Songs already listened to: 1348


In [21]:
def validate_recommendations(
    recommendations,
    already_listened
):
    overlap = set(
        recommendations["item_id"]
    ).intersection(already_listened)


    if len(overlap) > 0:
        return f"Invalid - overlapping songs: {overlap}"
    else:
        return "Valid"